In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import string

In [7]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.5938.62 Safari/537.36'
}


url = 'https://www.fragrantica.com/designers/'

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
else:
    print(f"Failed to retrieve page, status code: {response.status_code}")

NameError: name 'requests' is not defined

In [8]:
elements = soup.find_all('div', class_="designerlist cell small-6 large-4")

In [9]:
pop_designers = {}

for element in elements:
    
    link = element.find('a')
        
    url = link.get('href')
    designer = link.text.strip()
    
    uri = 'https://www.fragrantica.com' + url
    
    pop_designers[designer] = uri

In [10]:
pop_designers

{'Acqua di Parma': 'https://www.fragrantica.com/designers/Acqua-di-Parma.html',
 'Afnan': 'https://www.fragrantica.com/designers/Afnan.html',
 'Al Haramain Perfumes': 'https://www.fragrantica.com/designers/Al-Haramain-Perfumes.html',
 'Al-Rehab': 'https://www.fragrantica.com/designers/Al-Rehab.html',
 'Amouage': 'https://www.fragrantica.com/designers/Amouage.html',
 'Ariana Grande': 'https://www.fragrantica.com/designers/Ariana-Grande.html',
 'Armaf': 'https://www.fragrantica.com/designers/Armaf.html',
 'Avon': 'https://www.fragrantica.com/designers/Avon.html',
 'Azzaro': 'https://www.fragrantica.com/designers/Azzaro.html',
 'BDK Parfums': 'https://www.fragrantica.com/designers/BDK-Parfums.html',
 'Bath & Body Works': 'https://www.fragrantica.com/designers/Bath-Body-Works.html',
 'Bond No 9': 'https://www.fragrantica.com/designers/Bond-No-9.html',
 'Britney Spears': 'https://www.fragrantica.com/designers/Britney-Spears.html',
 'Burberry': 'https://www.fragrantica.com/designers/Burberry

In [14]:
perfume_info = []

for designer, url in pop_designers.items():
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.content, 'html.parser')
    
    perfume_links = soup.find_all('div', class_="flex-child-auto")

    for perfume_link in perfume_links:
        tag = perfume_link.find('a')
        
        perfume_url = tag['href']
        perfume = tag.text.strip()
        
        perfume_uri = f'https://www.fragrantica.com{perfume_url}'
        
        perfume_info.append({
            'Designer': designer,
            'Perfume URL': perfume_uri  # Join accords into a single string
        })
        

In [15]:
len(perfume_info)

2602

---
### Collecting Note Categories and Notes

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

url = 'https://www.fragrantica.com/notes/'
response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
else:
    print(f"Failed to retrieve page, status code: {response.status_code}")

In [3]:
elements = soup.find('div', class_="cell text-center gone4empty")

In [9]:
tags = elements.find_all('a')

In [24]:
categories = []

for item in tags: 
    category = item.get_text()
    categories.append(category)

In [25]:
categories

['CITRUS SMELLS',
 'FRUITS, VEGETABLES AND NUTS',
 'FLOWERS',
 'WHITE FLOWERS',
 'GREENS, HERBS AND FOUGERES',
 'SPICES',
 'SWEETS AND GOURMAND SMELLS',
 'WOODS AND MOSSES',
 'RESINS AND BALSAMS',
 'MUSK, AMBER, ANIMALIC SMELLS',
 'BEVERAGES',
 'NATURAL AND SYNTHETIC, POPULAR AND WEIRD']

In [68]:
categories_notes = {}

for i in range(1,13):
    if i<10:
        category_id = f'groupnotes_group_0{i}_title'
    else:
        category_id = f'groupnotes_group_{i}_title'

    category_div = soup.find('div', id=category_id)
    
    if category_div:
        
        category_name = category_div.text.strip()
        
        descs_container = category_div.find_next('div').find_next('div')
        category_descriptions = descs_container.get_text()
        #print(category_descriptions)
        
        notes_container = category_div.find_next('div').find_next('div').find_next('div').find_next('div')
        
        categories_notes[category_name] = {
            'description': category_descriptions,
            'notes': []
        }
        
        if notes_container:
            notes = [note.text.strip() for note in notes_container.find_all('div', 
                                                                            class_='cell small-6 medium-4 large-3 text-center notebox')]
            #print(notes)
            
            categories_notes[category_name]['notes'] = notes
        


In [1]:
#categories_notes

In [78]:
notes_data = pd.DataFrame({
    'Category': list(categories_notes.keys()),  # Get category names as a list
    'Description': [data['description'] for data in categories_notes.values()],  # Get descriptions
    'Notes': [', '.join(data['notes']) for data in categories_notes.values()]  # Join notes into a single string
})

In [79]:
notes_data

,Category,Description,Notes
0,CITRUS SMELLS,\nMost often by citrus in perfumery we describ...,"Bergamot, Bigarade, Bitter Orange, Blood Orang..."
1,"FRUITS, VEGETABLES AND NUTS",\nFruity notes beyond citrus (which form a cla...,"Acai Berry, Acerola, Acorn, Almond, Apple, App..."
2,FLOWERS,"\nA self-evident category of fragrance notes, ...","Acacia, Acerola Blossom, Aglaia, Albizia, Almo..."
3,WHITE FLOWERS,"\nThis is a subgroup within the Flowers group,...","Arum Lily, Belanis, Black Locust, Boronia, Car..."
4,"GREENS, HERBS AND FOUGERES","\nBy the term ""green"" we refer to notes of sna...","Agave, Algae, Aloe Vera, Ammophila (Beach Gras..."
5,SPICES,\nThe Spices group is a familiar category of p...,"Allspice, Anise, Asafoetida, Bay Leaf, Bengal ..."
6,SWEETS AND GOURMAND SMELLS,\nThis succulent group of scent notes has real...,"Acetyl Furan, Agave Nectar, Apple Pie, Baba (I..."
7,WOODS AND MOSSES,"\nWoody notes are dependable and pliable, a so...","Agarwood (Oud), Akigalawood, Alder, Almond tre..."
8,RESINS AND BALSAMS,\nThe raw materials falling under the umbrella...,"Amberwood, Andiroba, Bakhoor, Balsamic Notes, ..."
9,"MUSK, AMBER, ANIMALIC SMELLS","\nThe term ""animalic"" refers to both raw aroma...","Akashic Acord, Aldron, Amber, Amber Xtreme, Am..."


In [80]:
notes_data.to_csv('categories_notes.csv', index=False)

---
## Getting all Designers

In [4]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}


In [5]:
letters = list(string.ascii_uppercase)
letter_to_number = {'A':1, 'B':2, 'C':3, 'D':4, 'E':4,
                     'F':5  , 'G':5, 'H':5, 'I':6, 'J':6, 'K':6, 'L':7, 
                     'M':8, 'N':9, 'O':9, 'P':9, 'Q':9, 'R':10, 'S':10, 
                     'T':11, 'U':11, 'V':11, 'W':11, 'X':11, 'Y':11, 'Z':11} 

for letter in letters:
    number = letter_to_number[letter]  # Get the corresponding number
    url = f'https://www.fragrantica.com/designers-{number}/#{letter}'
    #print(url)
    
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.content, "html.parser")
    
    elements = soup.find_all('div', class_="designerlist cell small-6 large-4")
    
    
    designers = {}

    for element in elements:

        link = element.find('a')

        url = link.get('href')
        designer = link.text.strip()

        uri = 'https://www.fragrantica.com' + url

        designers[designer] = uri


---
### Tests

In [7]:
url = 'https://www.fragrantica.com/designers/Glossier.html'

page = requests.get(url, headers=headers)
soup = BeautifulSoup(page.content, 'html.parser')

In [19]:
elements = soup.find_all('div', class_="flex-child-auto")

for element in elements:
    tag = element.find('a')
    
    link = tag.get('href')
    perfume = tag.text.strip()
    print(link, perfume)

/perfume/Glossier/You-46885.html You
/perfume/Glossier/You-Doux-97605.html You Doux
/perfume/Glossier/You-Reve-97604.html You Rêve
